In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import re
import glob 

On associe une province à chaque Ping en bouclant sur les fichier provinces

In [ ]:
ds_sv_path = Path("/data/rd_exchange/salbernhe/data/acoustics/profiles38kHz10t0750m20251006.nc")


ds_sv = xr.open_dataset(ds_sv_path)
ds_sv = ds_sv.assign_coords(
    year=("time", ds_sv["time"].dt.year.values),
    month=("time", ds_sv["time"].dt.month.values)

)

idx_ping_prov = np.full(ds_sv.dims["time"], np.nan, dtype=object)

prov_folder = sorted(glob.glob("/data/rd_exchange/asauvebois/moy_month/*_monthly_mean.nc"))

In [ ]:
#1. On boucle sur les fichies provinces

for file_prov in prov_folder:
    nom = file_prov.split("/")[-1].replace(".nc", "")   # "2003_12_monthly_mean"

    year_str = int(file_prov.split("_")[0])
    month_str = int(file_prov.split("_")[1])

    mask= (ds_sv["year"]==year_str) & (ds_sv["month"]==month_str)
    if not mask.any():
        continue
   
    # on va chercher le point de grille le plus proche 

    sub = ds_sv.sel(time=mask) #on applique le mask qui filtre le ds_sv et ne garde que les ping du mois/année en cours
    ds_prov = xr.open_dataset(file_prov) #on ouvre le fichier de province correspondant
    prov_sel = ds_prov["prov"].sel(latitude=sub["latitude"], longitude=sub["longitude"], method="nearest") #on récupère le fameux point de grille

    idx_ping_prov[mask.values] = prov_sel.values #on ajoute au tableau la province correspondant aux coordonnées pour le mois considéré
    ds_prov.close()
    

ds_sv = ds_sv.assign_coords(province=("time", idx_ping_prov)) #on ajoute la province au dataset acoustique

    # --- Vérification rapide ---
n_missing = np.sum(idx_ping_prov == None) if idx_ping_prov.dtype == object else np.isnan(idx_ping_prov.astype(float)).sum()
print(f"Pings sans province assignée : {n_missing} / {ds_sv.dims['time']}")
print("Provinces uniques trouvées :", np.unique([p for p in idx_ping_prov if p is not None]))







In [ ]:
#On passe en linéaire pour pouvoir faire les calculs de moyenne


